# 模型推理 - 使用 QLoRA 微调后的 ChatGLM4-9B

In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HOME'] = '/hy-tmp'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import pandas as pd
import json

# 定义全局变量和参数
model_name_or_path = 'THUDM/glm-4-9b-chat-hf'  # 模型ID或本地路径
model_output_dir = "./model_output/self_data_chatglm4_zhouyi_lora"

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1、加载base model

In [2]:
bnb_config = BitsAndBytesConfig(load_in_4bit = True,
                              bnb_4bit_quant_type = 'nf4',
                              bnb_4bit_use_double_quant = True,
                              bnb_4bit_compute_dtype = torch.bfloat16)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model.requires_grad_(False)
base_model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:33<00:00,  8.26s/it]


GlmForCausalLM(
  (model): GlmModel(
    (embed_tokens): Embedding(151552, 4096, padding_idx=151329)
    (layers): ModuleList(
      (0-39): 40 x GlmDecoderLayer(
        (self_attn): GlmAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=True)
          (k_proj): Linear4bit(in_features=4096, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=4096, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): GlmMLP(
          (gate_up_proj): Linear4bit(in_features=4096, out_features=27392, bias=False)
          (down_proj): Linear4bit(in_features=13696, out_features=4096, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): GlmRMSNorm((4096,), eps=1.5625e-07)
        (post_attention_layernorm): GlmRMSNorm((4096,), eps=1.5625e-07)
      )
    )
    (norm): GlmRMSNorm((4096,), eps=1.5625e-07)
    (rotary_emb): GlmRotaryEmbedding()
  )

In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('THUDM/glm-4-9b-chat-hf')

## 2、加载微调的模型

In [4]:
from peft import PeftModel, PeftConfig

peft_after_model = PeftModel.from_pretrained(base_model, model_output_dir)

In [5]:
peft_after_model.print_trainable_parameters()

trainable params: 0 || all params: 9,423,749,120 || trainable%: 0.0


## 4、抽样观察baseModel和peftModel输出结果

In [6]:
my_question = "我最近总是失眠，如果用《周易》的智慧来看，我该关注哪个卦象？"
# my_question = "周易要怎么理解和应用？算卦算命？还是一种古代的为人处世的方法论？"
message = [
    {
        "role": "system",
        "content": "你是一个精通《周易》的国学大师，帮助人们答疑解惑。"
    },
    {
        "role": "user",
        "content": {my_question}
    }
]
print("\nmessage", message)

inputs = tokenizer.apply_chat_template(
    message,
    return_tensors='pt',
    tokenize = True, 
    add_generation_prompt = True,
    return_dict=True,
).to('cuda')
input_len = inputs['input_ids'].shape[1]
generate_kwargs = {
    "input_ids": inputs['input_ids'],
    "attention_mask": inputs['attention_mask'],
    "max_new_tokens": 512,
    "do_sample": False,
}

peft_model_out = peft_after_model.generate(**generate_kwargs)
print("\npeft_model:",tokenizer.decode(peft_model_out[0][input_len:], skip_special_tokens=True))

print('\n----------------------------------------------------------------------\n')
## base_model得主动先把LORA部件给关闭
with peft_after_model.disable_adapter():
    base_model_out = base_model.generate(**generate_kwargs)
    print("\nbase_model:",tokenizer.decode(base_model_out[0][input_len:], skip_special_tokens=True))



message [{'role': 'system', 'content': '你是一个精通《周易》的国学大师，帮助人们答疑解惑。'}, {'role': 'user', 'content': {'我最近总是失眠，如果用《周易》的智慧来看，我该关注哪个卦象？'}}]

peft_model: 
失眠可对应《周易》的‘坎’卦。坎卦上坎下坎，象征水在水中，暗示内心动荡不安。卦辞‘有孚，威如’提示：保持诚信（有孚）可安定心神，威严（威如）指自我约束。象传‘水’喻情绪，‘险’喻压力，故需‘君子以常德行’——保持日常德行（如规律作息、冥想）以平心。邵雍解‘身体：肾水不足’提示可能肾虚，需注意饮食（咸味）与运动。傅佩荣解‘事业：险阻重重’暗示工作或生活压力可能加剧失眠，故需主动调整。

----------------------------------------------------------------------


base_model: 
根据您的情况，失眠可能是由于心神不宁、情绪波动等原因引起的。在《周易》中，与心神、情绪相关的卦象有很多，以下是一些建议关注的卦象：

1. **否卦（否）**）：否卦象征着逆境和困难，失眠可能是内心困扰的体现。

2. **既济卦（既济）**）：既济卦象征着事情已经完成，但可能因为某些原因导致心神不宁。

3. **未济卦（未济）**）：未济卦象征着事情尚未完成，可能因为内心的焦虑和不安导致失眠。

在关注这些卦象的同时，也可以结合个人的实际情况，如生活习惯、工作压力等，来调整自己的心态和生活节奏，从而可能改善失眠的状况。
